In [ ]:
"""
Compute HRV/PRV Features for all signals and after Window Removal

This script computes Heart Rate Variability (HRV) and Pulse Rate Variability (PRV)
metrics from ECG and PPG signals, starting from the detected peaks.

Main functionalities:
- Computes HRV and PRV features using 5-minute analysis windows with 30% overlap
  across the entire signal.
- Implements a noise removal procedure based on user-defined parameters
  (window duration, number of allowed peaks in |HR-PR| difference and number of 
  outliers per window).
- Automatically identifies and removes noisy or unreliable segments of the signal.
- Recomputes HRV and PRV metrics after window removal to obtain “cleaned” results.
- Visualizes the evolution of HRV and PRV metrics across the analysis windows,
  before and after the removal process.

This allows for a direct comparison between HRV and PRV trends and helps evaluate 
the impact of noise and artifacts on variability measures.
"""


# =============================================================================
# IMPORT LIBRARIES
# =============================================================================

import os
import numpy as np

from utils_remove_windows import (
    align_pr_to_hr, 
    compute_peaks_difference,
    compute_mean_std,
    extract_good_windows
)
from utils_compute_metrics import compute_metrics


%matplotlib qt


# =============================================================================
# DEFINE PATHS
# =============================================================================

# Path of npz file
path = 'C:/Users/ilari/Desktop/Sleep disorders/Example files/Database Paper/Database Paper - Peaks/Pz 13.npz'

# Path to save features 
path_hrv_all_windows = 'C:/Users/ilari/Documents/GitHub/SleepProject/Script/Script Paper/Data/All Windows/HRV'
path_prv_all_windows = 'C:/Users/ilari/Documents/GitHub/SleepProject/Script/Script Paper/Data/All Windows/PRV'

path_hrv_removal = 'C:/Users/ilari/Documents/GitHub/SleepProject/Script/Script Paper/Data/Window Removal/HRV'
path_prv_removal = 'C:/Users/ilari/Documents/GitHub/SleepProject/Script/Script Paper/Data/Window Removal/PRV'


# =============================================================================
# DEFINE VARIABLES
# =============================================================================

# Parameters for Window Removal
# - window_sec: duration (in seconds) of the analysis window
# - th_n_peaks: maximum number of |HR - PR| peaks allowed in a window
# - th_n_outliers: maximum number of HR/PR outliers allowed in a window (based on mean ± 3*std)
# - h_min: minimum height threshold for |HR - PR| peak detection
window_sec = 10
th_n_peaks = 1
th_n_outliers = 1
h_min = 20      

# Parameters for windows in HRV/PRV metrics
window_sec_metrics = 5 * 60     # Window size (in seconds): 5 minutes 
overlap = 0.3                   # Overlap percentage 
step_sec = int(window_sec_metrics * (1 - overlap))  # Step size (in seconds) for the sliding window


subj = os.path.splitext(os.path.basename(path))[0]
file_name = f'{subj}.csv' 


# =============================================================================
# LOAD DATA
# =============================================================================

data = np.load(path)

sleep_ecg = data["ecg"]
ecg_peaks_final = data["ecg_peaks"]
fs_ecg = data["ecg_sampling_rate"]

sleep_ppg = data["ppg"]
ppg_peaks_final = data["ppg_peaks"]
fs_ppg = data["ppg_sampling_rate"]


# =============================================================================
# ALIGNMENT PULSE RATE TO HEART RATE 
# =============================================================================

# Using cross-correlation to estimate and correct temporal delay (lag)
hr_final, pr_final, time_hr, time_pr, time_shift = align_pr_to_hr(
    ecg_peaks_final, ppg_peaks_final, fs_ecg, fs_ppg, len(sleep_ecg)
)


# =============================================================================
# COMPUTE FEATURES FOR ALL WINDOWS
# =============================================================================

# Compute HRV and PRV metrics for All Windows
df_hrv_all, df_prv_all = compute_metrics(
    hr_final, pr_final, time_hr, time_pr, window_sec_metrics, step_sec
)

# Saving features in csv (All Windows)
save_path_ecg = os.path.join(path_hrv_all_windows, file_name)
save_path_ppg = os.path.join(path_prv_all_windows, file_name)
#df_hrv_all.to_csv(save_path_ecg, index=False)
#df_prv_all.to_csv(save_path_ppg, index=False)



# =============================================================================
# BAD WINDOW DETECTION PARAMETERS AND WINDOW REMOVAL
# =============================================================================

# Compute |HR - PR| difference and detect peaks above a minimum threshold (h_min)
# These peaks indicate mismatched or noisy intervals between HR and PR signals
time_common, diff_hr_pr, diff_peaks = compute_peaks_difference(
    hr_final, pr_final, time_hr, time_pr, h_min
)

# Compute HR and PR upper/lower bounds (mean ± 3*std)
# Used to detect outliers in the HR and PR signals
hr_bounds, pr_bounds = compute_mean_std(hr_final, pr_final)


# Extract and reconstruct "clean" HR and PR segments based on outlier thresholds and |HR–PR| peaks count
hr_clean, pr_clean, time_hr_clean, time_pr_clean, windows_ranges, good_windows, n_peaks_per_window = extract_good_windows(
    hr_final, time_hr, pr_final, time_pr,
    diff_hr_pr, time_common, diff_peaks, 
    hr_bounds, pr_bounds, window_sec, th_n_outliers, th_n_peaks
)


# =============================================================================
# COMPUTE FEATURES AFTER WINDOW REMOVAL
# =============================================================================

# Calculating HRV and PRV features for the current case
df_hrv_good, df_prv_good = compute_metrics(
    hr_clean, pr_clean, time_hr_clean, time_pr_clean, window_sec_metrics, step_sec
)

# Saving features in csv (after Window Removal)
save_path_ecg = os.path.join(path_hrv_removal, file_name)
save_path_ppg = os.path.join(path_prv_removal, file_name)
#df_hrv_good.to_csv(save_path_ecg, index=False)
#df_prv_good.to_csv(save_path_ppg, index=False)


Lag: 95 samples (0.371 s)
Window 1: from 0s to 300s → 304 samples HR and 304 samples PR
Window 2: from 210s to 510s → 306 samples HR and 306 samples PR
Window 3: from 420s to 720s → 308 samples HR and 308 samples PR
Window 4: from 630s to 930s → 308 samples HR and 308 samples PR
Window 5: from 840s to 1140s → 310 samples HR and 310 samples PR
Window 6: from 1050s to 1350s → 314 samples HR and 313 samples PR
Window 7: from 1260s to 1560s → 316 samples HR and 315 samples PR
Window 8: from 1470s to 1770s → 315 samples HR and 315 samples PR
Window 9: from 1680s to 1980s → 320 samples HR and 320 samples PR
Window 10: from 1890s to 2190s → 322 samples HR and 322 samples PR
Window 11: from 2100s to 2400s → 322 samples HR and 321 samples PR
Window 12: from 2310s to 2610s → 324 samples HR and 324 samples PR
Window 13: from 2520s to 2820s → 321 samples HR and 321 samples PR
Window 14: from 2730s to 3030s → 321 samples HR and 321 samples PR
Window 15: from 2940s to 3240s → 320 samples HR and 320 